# Customer-Managed Keys (CMK) — how they work

This journey is an explainer and configuration reference, not a runnable lab. Customer-Managed Keys (CMK) are configured at the workspace and instance level by administrators. Once configured, Lakebase uses your keys transparently; you query normally.

This notebook teaches the concepts and shows you how to verify your configuration. If you are not a workspace admin, share this with your admin and return after CMK is enabled.

## What CMK is

Encryption at rest means your data on disk is encrypted. By default, Databricks (or your cloud provider) holds the encryption keys. With CMK, you hold your own keys in your KMS (Key Management Service):

- **AWS:** AWS KMS (Key Management Service)
- **Azure:** Azure Key Vault
- **GCP:** Google Cloud KMS

Databricks requests a decrypt operation from your KMS for each read. If you revoke the key, Databricks cannot decrypt, and all access stops immediately. It is your kill-switch.

## Why CMK matters

In regulated industries (healthcare, finance, government), your compliance requirements may mandate:

- You must control the encryption keys (no third-party key holders).
- You must be able to revoke access instantly.
- You must audit who accessed the keys and when.
- You must rotate keys on a schedule you define.

CMK lets you meet all of these in Lakebase.

## Latency impact

Each query makes decrypt calls to your KMS. This adds a few milliseconds per request:

- Direct Lakebase query: typically 10-50 ms round trip. With CMK, add 5-10 ms for KMS calls.
- Multi-round-trip workloads: if your workload makes many small queries, KMS latency compounds.

In practice, for most workloads, the overhead is unnoticeable. For extremely latency-sensitive applications (sub-millisecond), measure with CMK enabled in a staging environment first.

## How to set it up (admin task)

### AWS KMS

1. Create a symmetric encryption key in AWS KMS in the region where your workspace is deployed.
2. Attach a key policy that allows the Databricks service role (provided during account setup) to call Decrypt, GenerateDataKey, and DescribeKey.
3. Give the key ARN to your Databricks account team. They configure the workspace.
4. All new Lakebase instances in that workspace use the key from that point on.

### Azure Key Vault

1. Create a key in Azure Key Vault.
2. Grant the Databricks managed identity Decrypt, Encrypt, and other required permissions.
3. Provide the key URI to your Databricks account team.

### GCP Cloud KMS

1. Create a key in Cloud KMS.
2. Grant the Databricks service account the cloudkms.cryptoKeyEncrypter and cloudkms.cryptoKeyDecrypter roles.
3. Provide the key resource ID to your Databricks account team.

## Testing CMK (non-destructive)

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'common', 'scripts'))

from lakebase import connect

# A normal query. If CMK is enabled, Lakebase decrypts using your KMS in the background.
# You will not see the KMS call; it is transparent.

print("If CMK is enabled on your instance:")
print("  - This query will work normally.")
print("  - Behind the scenes, Lakebase called your KMS to decrypt the data pages.")
print("  - Check your KMS audit logs to see the decrypt calls.")
print()

try:
    with connect() as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT 1")
            result = cur.fetchone()
            print(f"Query result: {result}")
            print("Connection successful.")
except Exception as e:
    print(f"Error: {e}")
    print("If you see 'KMS permission denied', your KMS or key configuration has an issue.")

## Key rotation

### Automatic rotation

Enable automatic key rotation in your KMS (typically annual). Your KMS generates new key versions; Lakebase uses the latest for new writes. Old data is decrypted with old key versions until it is rewritten.

### Manual rotation

If you want to rotate immediately:

1. Create a new key version in your KMS.
2. Contact your Databricks account team to update the workspace config to the new key version.
3. Lakebase uses the new key for new writes. Old data is gradually re-encrypted during compaction.

## Revoking the key (kill-switch)

To immediately stop all access to encrypted data:

1. In your KMS, **disable** or **delete** the key.
2. Lakebase instances encrypted with that key will fail on the next query with a KMS permission error.
3. To restore access, re-enable the key in your KMS.

This is your compliance kill-switch: no Lakebase API call needed, just your KMS controls.

## Troubleshooting

### Query fails with 'KMS permission denied'

Check:
- Does the Databricks service role/identity still have permissions on the key in your KMS?
- Is the key enabled (not disabled or deleted)?
- Is the key in the same region as Lakebase?
- If using AWS cross-region replication, check that the replica key is also enabled.

### Decryption latency is high

Each decrypt call to your KMS adds a few milliseconds. If you see unusually high latency:
- Check your KMS service health in the cloud console.
- Verify network connectivity between your Databricks region and your KMS region.
- Consider enabling KMS request batching or caching (if your KMS provider supports it).
- Measure in a non-production environment to set baselines.

### CMK settings not visible in workspace console

CMK is available only on:
- Enterprise-tier Databricks workspaces.
- Supported cloud providers (GA on AWS and Azure, limited availability on GCP).
- Workspaces provisioned after CMK enablement.

Contact your Databricks account team to confirm availability.

## Security best practices

1. **Audit and log:** Enable and monitor KMS audit logs for all access and rotations.
2. **Least privilege:** Limit who can manage keys in your KMS. Use IAM roles / RBAC strictly.
3. **Automate rotation:** Set up automatic key rotation in your KMS (typically annual).
4. **Test recovery:** Periodically test revoking and re-enabling the key in a non-production environment to verify your runbook.
5. **Document:** Document your key management process, rotation schedule, and disaster recovery steps.
6. **Separate keys for environments:** Use different keys for dev, staging, and production. This limits blast radius if a key is compromised.
7. **Backup:** If your KMS supports it, set up key material backup (some KMS providers offer this for recovery).

## What you learned

CMK gives you full cryptographic control: your keys in your KMS, your rotation policy, your revocation kill-switch. Once enabled, CMK is invisible to users; queries work the same way, but with the confidence that your data is encrypted under your control. This is typically required in regulated industries and a security best practice for all organizations. For questions on your specific setup, contact your Databricks account team.